In [33]:
# Enable autoreload when modifying files
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
# Libraries
import pandas as pd

import duckdb
from pathlib import Path

from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.config.schema import load_schema_from_yaml
from icare_risk.clinphen.registry.registry import load_phenotypes_from_yaml
from icare_risk.clinphen.utils.history import build_historical_events_table
from icare_risk.clinphen.utils.history import evaluate_temporal_phenotypes
from icare_risk.utils.io import load_pkg_yaml, get_pkg_path

def compute_fm_batch(df_epi, df_problems):
    """"""
    # Load phenotypes config file
    config = load_pkg_yaml('config/icare/phenotypes.yaml')

    # Select Charlson phenotype entries
    config = {k:v for k,v in config.items() if  k.startswith('charlson_')}

    # Create auxiliary dataframe with first occurrence
    df_first_occ = build_historical_events_table(
        df=df_problems,
        configs=config,
        subject_col='SUBJECT',
        time_col='PROBLEM_DT_TM',
        code_col='PROBLEM_CODE'
    )
    display(df_first_occ.head(5))

    # 2. Enforce the temporal boundary against your active encounters
    # (Assuming you have an `episodes_df` containing SUBJECT and index_admission)
    temporally_valid_matrix = evaluate_temporal_phenotypes(
        episodes_df=df_epi,
        first_occurrences=df_first_occ,
        subject_col='SUBJECT',
        admission_col='ADMISSION_DATE',
        prefix='batch_'
    )

    # Return
    return temporally_valid_matrix


In [50]:
dir = Path('/app/notebooks/data/synthetic/10000p/')
print(f"Using data from: {dir.name}")

# 3. Load problems data
df_episodes = pd.read_csv(dir / 'icare_episodes_anon.csv', parse_dates=['ADMISSION_DATE'])
df_problems = pd.read_csv(dir / 'icare_problems_anon.csv', parse_dates=['PROBLEM_DT_TM'])

Using data from: 10000p


In [53]:
res1 = compute_fm_batch(df_episodes, df_problems)

# 3. Count unique patients safely
patient_counts = res1 \
    .drop(columns=["ADMISSION_DATE"]) \
    .groupby("SUBJECT").any().sum() \
    .reset_index(name='patient_count') \
    .rename(columns={"index": "event_name"}) \
    .sort_values('patient_count', ascending=False)

display(patient_counts)

,SUBJECT,event_name,first_occurrence_date
0,10003,charlson_hx_mi,2021-01-13
1,10003,charlson_hx_mi_icare,2021-01-13
2,10004,charlson_hx_cancer_solid,2023-05-05
3,10008,charlson_hx_liver_mild,2020-04-23
4,10012,charlson_hx_pulmonary,2023-09-11


,event_name,patient_count
0,SPELL_IDENTIFIER,10000
1,ENCNTR_ID,10000
2,EPISODE_IDENTIFIER,10000
3,ADMISSION_TIME,10000
4,AGE_AT_ADMISSION,10000
5,DISCHARGE_DATE,10000
6,MAIN_SPECIALTY_CODE,10000
7,MAIN_SPECIALTY_CODE_DESC,10000
8,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,10000
15,batch_charlson_hx_diabetes_uncomp,421


In [54]:
# Library
import duckdb
from icare_risk.clinphen.engine.runner import FeatureMatrixBuilder

# Define paths
schema_cfg = get_pkg_path('config/icare/schema.yaml')
phenotype_cfg = get_pkg_path('config/icare/phenotypes.yaml')

# Load schema and phenotypes definition
schema = load_schema_from_yaml(schema_cfg)
registry = load_phenotypes_from_yaml(phenotype_cfg)

# Create connection source
source = DuckDBSource(connection=duckdb.connect())

# Get specific phenotypes
charlson = registry.select_by_prefix('charlson', return_names=True)

# Create matrix
builder = FeatureMatrixBuilder(
    schema=schema, source=source,
    registry=registry,
    phenotypes=charlson
)

feature_matrix = builder.build(raise_on_error=True, show_progress=True)



Skipping 'definitions': missing module or function definition.
Preloading 3 domain tables (['diagnosis', 'prescribing', 'problems'] for 10000 subjects...
Loaded clinical domains into cache in 5.55s
Loaded clinical domains in 5.61s


Processing Episodes: 100%|██████████| 10000/10000 [00:59<00:00, 169.37it/s]


In [55]:
# Count unique patients safely
patient_counts = feature_matrix \
    .groupby("subject").any().sum() \
    .reset_index(name='patient_count') \
    .rename(columns={"index": "event_name"}) \
    .sort_values('patient_count', ascending=False)

# Show
display(patient_counts.head(10))

/tmp/ipykernel_11/2077594599.py:3: FutureWarning: 'any' with datetime64 dtypes is deprecated and will raise in a future version. Use (obj != pd.Timestamp(0)).any() instead.
  .groupby("subject").any().sum() \


,event_name,patient_count
0,admission_date,10000
1,admission_time,10000
2,discharge_date,10000
3,index_admission,10000
4,index_discharge,10000
18,charlson_hx_diabetes_uncomp,421
22,charlson_hx_cancer_solid,129
20,charlson_hx_renal_mod_sev,120
26,charlson_hx_cancer_met,86
14,charlson_hx_pulmonary,79


In [56]:
import pandas as pd

def compare_phenotype_counts(
    df_standard: pd.DataFrame,
    df_batch: pd.DataFrame,
    phenotype_names: list[str],
    batch_prefix: str = "batch_",
    subject_col: str = "subject"
) -> pd.DataFrame:
    """
    Creates a side-by-side comparison table of positive counts for both
    unique patients and total episodes between standard and batch runs.
    """
    # Reset index if subject is currently stored in the DataFrame index
    df_std = df_standard.reset_index() if subject_col not in df_standard.columns else df_standard
    df_bat = df_batch.reset_index() if subject_col not in df_batch.columns else df_batch

    # Normalize subject column name lookup to lowercase
    std_subj_col = next((c for c in df_std.columns if c.lower() == subject_col.lower()), None)
    bat_subj_col = next((c for c in df_bat.columns if c.lower() == subject_col.lower()), None)

    comparison_data = []

    for name in phenotype_names:
        std_col = name
        bat_col = f"{batch_prefix}{name}" if f"{batch_prefix}{name}" in df_bat.columns else name

        # --- Standard Matrix Counts ---
        std_episodes = 0
        std_patients = 0
        if std_col in df_std.columns:
            std_mask = df_std[std_col].fillna(0).astype(bool)
            std_episodes = std_mask.sum()
            if std_subj_col:
                std_patients = df_std.loc[std_mask, std_subj_col].nunique()

        # --- Batch Matrix Counts ---
        bat_episodes = 0
        bat_patients = 0
        if bat_col in df_bat.columns:
            bat_mask = df_bat[bat_col].fillna(0).astype(bool)
            bat_episodes = bat_mask.sum()
            if bat_subj_col:
                bat_patients = df_bat.loc[bat_mask, bat_subj_col].nunique()

        comparison_data.append({
            "Phenotype": name,
            "p_std": std_patients,
            "p_batch": bat_patients,
            "p_match": std_patients == bat_patients,
            "e_std": std_episodes,
            "e_batch": bat_episodes,
            "e_match": std_episodes == bat_episodes,
        })

    return pd.DataFrame(comparison_data)

In [57]:
# --- Run Side-by-Side Comparison ---
df_comparison = compare_phenotype_counts(
    df_standard=feature_matrix,
    df_batch=res1,
    phenotype_names=charlson,
    batch_prefix="batch_",
    subject_col="subject"
)

display(df_comparison)

,Phenotype,p_std,p_batch,p_match,e_std,e_batch,e_match
0,charlson_hx_mi,40,40,True,40,40,True
1,charlson_hx_mi_icare,44,40,False,44,40,False
2,charlson_hx_chf_code,55,55,True,55,55,True
3,charlson_hx_chf,55,55,True,55,55,True
4,charlson_hx_pvd,18,18,True,18,18,True
5,charlson_hx_pvd_icare,26,18,False,26,18,False
6,charlson_hx_stroke,46,46,True,46,46,True
7,charlson_hx_stroke_icare,50,46,False,50,46,False
8,charlson_hx_dementia,72,72,True,72,72,True
9,charlson_hx_pulmonary,79,79,True,79,79,True
